# Supplementary / Sensitivity Analyses

This notebook runs supplementary analyses for the bachelor's thesis:
post-hoc power analysis, mixed-effects models as sensitivity checks,
and extreme-reading exclusion robustness tests.

**Sections:**
1. Post-hoc power analysis (paired & independent designs)
2. Mixed-effects models (random-intercept, circadian-adjusted)
3. Sensitivity analysis — extreme reading exclusion
4. Summary of exported artifacts

In [ ]:
import sys
import os

# Ensure CWD is project root so Config relative paths resolve correctly
os.chdir(os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))
sys.path.insert(0, ".")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
import warnings

from src.config import Config, Columns
from src.data_processing import DataLoader
from analysis.thesis_stats import (
    variance_decomposition,
    compute_icc1,
    add_sin_cos_hour,
    export_table,
    bootstrap_ci,
)

config = Config()
THESIS_DIR = Path("results/thesis")
THESIS_DIR.mkdir(parents=True, exist_ok=True)
DPI = 300
sns.set_theme(style="whitegrid", font_scale=1.1)

In [ ]:
# Load all data sources
df = pd.read_parquet(THESIS_DIR / "monitoring_labeled.parquet")
df[Columns.TIME] = pd.to_datetime(df[Columns.TIME])
print(f"Monitoring data: {len(df)} rows, {df[Columns.PAT_ID].nunique()} subjects")

sc_medians = pd.read_csv(THESIS_DIR / "subject_context_medians.csv")
print(f"Subject-context medians: {len(sc_medians)} rows")

agg = pd.read_csv(THESIS_DIR / "aggregated_clean.csv")
print(f"Aggregated clean: {len(agg)} subjects")

res = pd.read_csv(config.get_results_path(config.SUBJECT_METRICS_OUTPUT))
print(f"Per-subject metrics: {len(res)} subjects")

---
## 1. Post-Hoc Power Analysis

We compute the minimum detectable effect sizes at 80% power for the sample
sizes available in this study. These are parametric approximations based on
the t-test; actual non-parametric power (Wilcoxon) is somewhat lower, so
the true detectable effects are slightly larger than reported here.

In [ ]:
from statsmodels.stats.power import TTestPower, TTestIndPower

# Paired (within-subject) power: detectable effect sizes
paired_power = TTestPower()
print("Paired comparisons — minimum detectable dz (alpha=0.05, power=0.80):")
paired_results = []
for n in [28, 27, 26, 11]:
    d = paired_power.solve_power(nobs=n, power=0.8, alpha=0.05)
    print(f"  N={n}: minimum detectable dz = {d:.3f}")
    paired_results.append({"Design": "Paired", "N": n, "Min detectable d": round(d, 3)})

print()
print(
    "Note: these are parametric (t-test) approximations. "
    "Actual non-parametric (Wilcoxon) power is somewhat lower, "
    "so the true minimum detectable effect sizes are slightly larger."
)

In [ ]:
# Independent groups power: detectable effect sizes for key subgroup splits
ind_power = TTestIndPower()
print("Independent groups — minimum detectable d (alpha=0.05, power=0.80):")
ind_results = []
for n1, n2, label in [
    (16, 12, "Male vs Female"),
    (11, 17, "Young vs Older"),
    (18, 10, "Healthy vs Diagnosed"),
]:
    d = ind_power.solve_power(nobs1=n1, ratio=n2 / n1, power=0.8, alpha=0.05)
    print(f"  {label} ({n1} vs {n2}): minimum detectable d = {d:.3f}")
    ind_results.append(
        {"Design": f"Independent ({label})", "N": f"{n1} vs {n2}", "Min detectable d": round(d, 3)}
    )

In [ ]:
# Power curve plot
effect_sizes = np.linspace(0.01, 1.5, 200)

fig, ax = plt.subplots(figsize=(9, 6))

# Paired N=28
power_28 = [paired_power.power(d, nobs=28, alpha=0.05) for d in effect_sizes]
ax.plot(effect_sizes, power_28, label="Paired N=28 (full cohort)", linewidth=2)

# Paired N=11 (Air Alert)
power_11 = [paired_power.power(d, nobs=11, alpha=0.05) for d in effect_sizes]
ax.plot(effect_sizes, power_11, label="Paired N=11 (Air Alert)", linewidth=2, linestyle="--")

# Independent N=16 vs 12
power_ind = [
    ind_power.power(d, nobs1=16, ratio=12 / 16, alpha=0.05) for d in effect_sizes
]
ax.plot(
    effect_sizes, power_ind,
    label="Independent 16 vs 12 (Male vs Female)", linewidth=2, linestyle="-.",
)

# Reference lines
ax.axhline(0.8, color="gray", linestyle=":", linewidth=1, label="Power = 0.80")
for bench, name in [(0.2, "small"), (0.5, "medium"), (0.8, "large")]:
    ax.axvline(bench, color="lightgray", linestyle=":", linewidth=0.8)
    ax.text(bench, 0.02, name, ha="center", va="bottom", fontsize=8, color="gray")

ax.set_xlabel("Effect size (Cohen's d / dz)")
ax.set_ylabel("Statistical power")
ax.set_title("Post-Hoc Power Analysis")
ax.set_xlim(0, 1.5)
ax.set_ylim(0, 1.02)
ax.legend(loc="lower right", fontsize=9)

plt.tight_layout()
fig.savefig(THESIS_DIR / "fig_S2_power_curves.png", dpi=DPI, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Saved: {THESIS_DIR / 'fig_S2_power_curves.png'}")

In [ ]:
# Summary table of power analysis results
def _interpret_d(d: float) -> str:
    if d < 0.2:
        return "trivial"
    if d < 0.5:
        return "small"
    if d < 0.8:
        return "medium"
    return "large"


power_rows = []
for r in paired_results:
    power_rows.append({
        "Design": "Paired (within-subject)",
        "N (or N1 vs N2)": str(r["N"]),
        "Min Detectable d (alpha=0.05, power=0.80)": r["Min detectable d"],
        "Interpretation": _interpret_d(r["Min detectable d"]),
    })
for r in ind_results:
    power_rows.append({
        "Design": r["Design"],
        "N (or N1 vs N2)": str(r["N"]),
        "Min Detectable d (alpha=0.05, power=0.80)": r["Min detectable d"],
        "Interpretation": _interpret_d(r["Min detectable d"]),
    })

power_table = pd.DataFrame(power_rows)
print(power_table.to_string(index=False))

export_table(power_table, THESIS_DIR / "table_S1_power_analysis")
print(f"\nSaved: {THESIS_DIR / 'table_S1_power_analysis.csv'}")

---
## 2. Mixed-Effects Models (Sensitivity Analysis)

We fit random-intercept linear mixed models as a sensitivity check for the
primary non-parametric results. These models account for the nested structure
(repeated measurements within subjects) and control for circadian variation.

**Interpretation caveat:** these are sensitivity analyses, not the primary
inferential results. The mixed models assume normally distributed residuals,
which may not hold for all outcomes.

In [ ]:
import statsmodels.formula.api as smf

# Prepare long data for mixed models
contexts_of_interest = [
    Columns.LABEL_BASELINE,
    Columns.LABEL_SLEEP,
    Columns.LABEL_COGNITIVE_TASK,
    Columns.LABEL_PHYSICAL_TASK,
]
df_mix = df[df[Columns.LABEL].isin(contexts_of_interest)].copy()
df_mix = add_sin_cos_hour(df_mix, datetime_col=Columns.TIME)
df_mix["label_cat"] = pd.Categorical(
    df_mix[Columns.LABEL],
    categories=[
        Columns.LABEL_BASELINE,
        Columns.LABEL_SLEEP,
        Columns.LABEL_COGNITIVE_TASK,
        Columns.LABEL_PHYSICAL_TASK,
    ],
)

print(f"Mixed model data: {len(df_mix)} readings from {df_mix[Columns.PAT_ID].nunique()} subjects")
print(f"\nContext distribution:\n{df_mix['label_cat'].value_counts().sort_index()}")

In [ ]:
# Random-intercept model for DBP
formula = "DBP ~ C(label_cat, Treatment(reference='Baseline')) + sin_hour + cos_hour"

with warnings.catch_warnings():
    warnings.simplefilter("always")
    model_dbp = smf.mixedlm(
        formula,
        data=df_mix,
        groups=df_mix[Columns.PAT_ID],
    )
    try:
        result_dbp = model_dbp.fit(reml=True)
        print(result_dbp.summary())
    except Exception as exc:
        print(f"WARNING: DBP model did not converge: {exc}")
        result_dbp = None

In [ ]:
# Random-intercept models for SBP and HR
result_sbp = None
result_hr = None

for outcome, name in [("SBP", "SBP"), ("HR", "HR")]:
    formula_out = (
        f"{outcome} ~ C(label_cat, Treatment(reference='Baseline')) + sin_hour + cos_hour"
    )
    with warnings.catch_warnings():
        warnings.simplefilter("always")
        model = smf.mixedlm(
            formula_out,
            data=df_mix,
            groups=df_mix[Columns.PAT_ID],
        )
        try:
            result = model.fit(reml=True)
            print(f"\n{'='*60}")
            print(f"  {name} Model")
            print(f"{'='*60}")
            print(result.summary())
            if outcome == "SBP":
                result_sbp = result
            else:
                result_hr = result
        except Exception as exc:
            print(f"WARNING: {name} model did not converge: {exc}")

In [ ]:
# Extract and format coefficients from all three models
coef_rows = []
for outcome_name, result in [("DBP", result_dbp), ("SBP", result_sbp), ("HR", result_hr)]:
    if result is None:
        continue
    summary_df = result.summary().tables[1]
    fe_params = result.fe_params
    bse = result.bse
    tvalues = result.tvalues
    pvalues = result.pvalues
    conf = result.conf_int()

    for param in fe_params.index:
        # Clean up predictor name for readability
        clean_name = (
            param.replace("C(label_cat, Treatment(reference='Baseline'))", "")
            .replace("[", "")
            .replace("]", "")
            .replace("T.", "")
        )
        if clean_name == "Intercept":
            clean_name = "Intercept (Baseline)"
        coef_rows.append({
            "Outcome": outcome_name,
            "Predictor": clean_name,
            "Coefficient": round(float(fe_params[param]), 3),
            "SE": round(float(bse[param]), 3),
            "z": round(float(tvalues[param]), 3),
            "p": round(float(pvalues[param]), 4),
            "95% CI": f"[{conf.loc[param, 0]:.2f}, {conf.loc[param, 1]:.2f}]",
        })

coef_table = pd.DataFrame(coef_rows)
if len(coef_table) > 0:
    print(coef_table.to_string(index=False))
    export_table(coef_table, THESIS_DIR / "table_S3_mixedlm_coefficients")
    print(f"\nSaved: {THESIS_DIR / 'table_S3_mixedlm_coefficients.csv'}")
else:
    print("No model results available to export.")

In [ ]:
# Marginal means visualization
# For each outcome, compute predicted marginal means per context
# holding circadian terms at their sample means

sin_mean = df_mix["sin_hour"].mean()
cos_mean = df_mix["cos_hour"].mean()

marginal_rows = []
for outcome_name, result in [("DBP", result_dbp), ("SBP", result_sbp), ("HR", result_hr)]:
    if result is None:
        continue
    fe = result.fe_params
    bse = result.bse
    intercept = fe["Intercept"]

    for ctx in contexts_of_interest:
        # Build predicted value
        pred = intercept
        # Add context effect (Baseline is reference = 0)
        for param_name in fe.index:
            if ctx != Columns.LABEL_BASELINE and ctx in param_name:
                pred += fe[param_name]
            elif "sin_hour" in param_name:
                pred += fe[param_name] * sin_mean
            elif "cos_hour" in param_name:
                pred += fe[param_name] * cos_mean

        # Approximate SE for marginal mean (simplified: SE of intercept + context coef)
        se_val = bse["Intercept"]
        for param_name in fe.index:
            if ctx != Columns.LABEL_BASELINE and ctx in param_name:
                se_val = np.sqrt(bse["Intercept"] ** 2 + bse[param_name] ** 2)
                break

        marginal_rows.append({
            "Outcome": outcome_name,
            "Context": ctx,
            "Marginal Mean": float(pred),
            "SE": float(se_val),
            "CI_low": float(pred - 1.96 * se_val),
            "CI_high": float(pred + 1.96 * se_val),
        })

marginal_df = pd.DataFrame(marginal_rows)

if len(marginal_df) > 0:
    outcomes = marginal_df["Outcome"].unique()
    fig, axes = plt.subplots(1, len(outcomes), figsize=(5 * len(outcomes), 5), sharey=False)
    if len(outcomes) == 1:
        axes = [axes]

    palette = sns.color_palette("Set2", n_colors=len(contexts_of_interest))

    for ax, outcome in zip(axes, outcomes):
        sub = marginal_df[marginal_df["Outcome"] == outcome]
        x_pos = range(len(sub))
        ax.errorbar(
            x_pos,
            sub["Marginal Mean"],
            yerr=1.96 * sub["SE"],
            fmt="o",
            capsize=5,
            capthick=1.5,
            markersize=8,
            color="steelblue",
            ecolor="gray",
        )
        ax.set_xticks(list(x_pos))
        ax.set_xticklabels(sub["Context"].values, rotation=30, ha="right", fontsize=9)
        units = "bpm" if outcome == "HR" else "mmHg"
        ax.set_ylabel(f"{outcome} ({units})")
        ax.set_title(f"{outcome} — Marginal Means")

    plt.tight_layout()
    fig.savefig(THESIS_DIR / "fig_S3_mixed_model_marginal_means.png", dpi=DPI, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Saved: {THESIS_DIR / 'fig_S3_mixed_model_marginal_means.png'}")
else:
    print("No marginal means to plot (models did not converge).")

In [ ]:
# Model diagnostics: QQ-plot and residuals vs fitted for DBP model
if result_dbp is not None:
    resid = result_dbp.resid
    fitted = result_dbp.fittedvalues

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # QQ-plot
    stats.probplot(resid, dist="norm", plot=ax1)
    ax1.set_title("QQ-Plot of DBP Model Residuals")
    ax1.get_lines()[0].set_markersize(3)

    # Residuals vs fitted
    ax2.scatter(fitted, resid, alpha=0.3, s=10, color="steelblue")
    ax2.axhline(0, color="red", linestyle="--", linewidth=1)
    ax2.set_xlabel("Fitted values")
    ax2.set_ylabel("Residuals")
    ax2.set_title("Residuals vs Fitted — DBP Model")

    # Shapiro-Wilk on a subsample (Shapiro-Wilk limited to N<=5000)
    resid_sample = resid.values
    if len(resid_sample) > 5000:
        rng = np.random.default_rng(42)
        resid_sample = rng.choice(resid_sample, size=5000, replace=False)
    w_stat, p_val = stats.shapiro(resid_sample)
    ax2.text(
        0.02, 0.98,
        f"Shapiro-Wilk: W={w_stat:.4f}, p={p_val:.4f}",
        transform=ax2.transAxes, va="top", fontsize=9,
        bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", alpha=0.8),
    )

    plt.tight_layout()
    fig.savefig(THESIS_DIR / "fig_S4_dbp_model_diagnostics.png", dpi=DPI, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Saved: {THESIS_DIR / 'fig_S4_dbp_model_diagnostics.png'}")
else:
    print("DBP model not available for diagnostics.")

---
## 3. Sensitivity Analysis — Extreme Reading Exclusion

We re-run the Baseline vs Cognitive Task comparison on DBP after excluding
physiologically extreme readings (SBP<70 or SBP>180, DBP<40 or DBP>120,
HR<45 or HR>150). This tests whether the primary results are robust to
potential measurement artifacts.

In [ ]:
# Define extreme reading thresholds
extreme_mask = (
    (df[Columns.SBP] < 70) | (df[Columns.SBP] > 180)
    | (df[Columns.DBP] < 40) | (df[Columns.DBP] > 120)
    | (df[Columns.HR] < 45) | (df[Columns.HR] > 150)
)
n_extreme = extreme_mask.sum()
print(f"Extreme readings removed: {n_extreme} / {len(df)} ({n_extreme / len(df) * 100:.1f}%)")

df_clean = df[~extreme_mask].copy()
print(f"Remaining readings: {len(df_clean)}")

# Recompute subject-context medians on clean data
sc_clean = (
    df_clean.groupby([Columns.PAT_ID, Columns.LABEL])
    .agg(
        SBP_med=(Columns.SBP, "median"),
        DBP_med=(Columns.DBP, "median"),
        HR_med=(Columns.HR, "median"),
        n=(Columns.SBP, "count"),
    )
    .reset_index()
)

# Get Baseline vs Cognitive Task paired data — original
baseline_orig = sc_medians[sc_medians[Columns.LABEL] == Columns.LABEL_BASELINE].set_index(
    Columns.PAT_ID
)
cog_orig = sc_medians[sc_medians[Columns.LABEL] == Columns.LABEL_COGNITIVE_TASK].set_index(
    Columns.PAT_ID
)
common_orig = baseline_orig.index.intersection(cog_orig.index)

# Get Baseline vs Cognitive Task paired data — sensitivity (clean)
baseline_clean = sc_clean[sc_clean[Columns.LABEL] == Columns.LABEL_BASELINE].set_index(
    Columns.PAT_ID
)
cog_clean = sc_clean[sc_clean[Columns.LABEL] == Columns.LABEL_COGNITIVE_TASK].set_index(
    Columns.PAT_ID
)
common_clean = baseline_clean.index.intersection(cog_clean.index)

print(f"\nOriginal paired N: {len(common_orig)}")
print(f"Sensitivity paired N: {len(common_clean)}")

# Wilcoxon signed-rank tests
comparison_rows = []

for dataset_label, bl, cg, common in [
    ("Original", baseline_orig, cog_orig, common_orig),
    ("Sensitivity (extremes removed)", baseline_clean, cog_clean, common_clean),
]:
    bl_vals = bl.loc[common, "DBP_med"].values
    cg_vals = cg.loc[common, "DBP_med"].values
    diff = cg_vals - bl_vals
    diff_nz = diff[diff != 0]

    if len(diff_nz) >= 5:
        w_result = stats.wilcoxon(diff_nz)
        n_eff = len(diff_nz)
        # Simple rank-biserial (matches bootstrap CI formula)
        ranks = stats.rankdata(np.abs(diff_nz))
        w_plus = float(np.sum(ranks[diff_nz > 0]))
        r_rb = 4 * w_plus / (n_eff * (n_eff + 1)) - 1 if n_eff > 0 else 0.0

        comparison_rows.append({
            "Dataset": dataset_label,
            "N pairs": len(common),
            "N non-zero diffs": n_eff,
            "Median diff (Cog - BL)": round(float(np.median(diff)), 2),
            "W statistic": round(float(w_result.statistic), 1),
            "p-value": round(float(w_result.pvalue), 4),
            "r_rb": round(float(r_rb), 3),
        })
    else:
        comparison_rows.append({
            "Dataset": dataset_label,
            "N pairs": len(common),
            "N non-zero diffs": len(diff_nz),
            "Median diff (Cog - BL)": round(float(np.median(diff)), 2) if len(diff) > 0 else np.nan,
            "W statistic": np.nan,
            "p-value": np.nan,
            "r_rb": np.nan,
        })

sensitivity_table = pd.DataFrame(comparison_rows)
print("\nBaseline vs Cognitive Task (DBP) — Original vs Sensitivity:")
print(sensitivity_table.to_string(index=False))

export_table(sensitivity_table, THESIS_DIR / "table_S4_sensitivity_extreme_exclusion")
print(f"\nSaved: {THESIS_DIR / 'table_S4_sensitivity_extreme_exclusion.csv'}")

---
## 4. Summary

In [ ]:
# Summary of all exported artifacts from this notebook
artifacts = [
    ("fig_S2_power_curves.png", "Power analysis curves for paired and independent designs"),
    ("table_S1_power_analysis.csv/.tex", "Minimum detectable effect sizes summary"),
    ("table_S3_mixedlm_coefficients.csv/.tex", "Mixed-model fixed-effect coefficients (DBP, SBP, HR)"),
    ("fig_S3_mixed_model_marginal_means.png", "Predicted marginal means per context from mixed models"),
    ("fig_S4_dbp_model_diagnostics.png", "QQ-plot and residuals vs fitted for DBP mixed model"),
    ("table_S4_sensitivity_extreme_exclusion.csv/.tex", "Sensitivity comparison: original vs extreme-excluded"),
]

print("Exported artifacts from this notebook:")
print("=" * 70)
for filename, description in artifacts:
    path = THESIS_DIR / filename.split("/")[0]
    exists = path.exists()
    status = "OK" if exists else "MISSING"
    print(f"  [{status}] {filename}")
    print(f"         {description}")

print("\nKey findings:")
print(
    "  1. Power analysis: with N=28 paired, the study can detect medium effects (dz >= ~0.55)."
)
print(
    "     For Air Alert (N=11), only large effects (dz >= ~0.94) are detectable."
)
print(
    "  2. Mixed-effects models provide a parametric sensitivity check, controlling"
)
print(
    "     for circadian variation and between-subject heterogeneity."
)
print(
    "  3. Extreme reading exclusion tests robustness of the primary DBP findings."
)